In [2]:
import sys
sys.path.append("..")

from src.solvers.equations_base import BaseEquationSolver

In [3]:
import re
from typing import List, Tuple, Dict, Optional

import pandas as pd
import statistics
from tqdm import tqdm

tqdm.pandas()


In [4]:
data = pd.read_csv("../data/raw/train.csv")

In [5]:
data["prompt_eda"] = data.prompt.str.split('.').apply(lambda x: x[0])

In [6]:
task_classes = {
    "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers":  "bit manipulation",
    "In Alice's Wonderland, secret encryption rules are used on text": "encryption",
    "In Alice's Wonderland, numbers are secretly converted into a different numeral system": "conversion to diff numeral system",
    "In Alice's Wonderland, a secret unit conversion is applied to measurements": "unit conversion",
    "In Alice's Wonderland, the gravitational constant has been secretly changed": "gravitational",
    "In Alice's Wonderland, a secret set of transformation rules is applied to equations": "equations transformation"
}

In [7]:
data["label"] = data.prompt_eda.map(task_classes)
data["label"].value_counts()

label
bit manipulation                     1602
gravitational                        1597
unit conversion                      1594
encryption                           1576
conversion to diff numeral system    1576
equations transformation             1555
Name: count, dtype: int64

In [8]:
eq_df = data[data['label'] == 'equations transformation'].copy()#.sample(50, random_state=1244124)

In [9]:
solver = BaseEquationSolver()

# 1. Функция для извлечения и классификации
def classify_prompt(prompt: str) -> str:
    examples, target = solver._extract_sections(prompt)
    if not examples:
        return "Extraction Failed"
    return solver._classify_task(examples, target)[0]

# Применяем классификацию ко всем промптам
eq_df['task_type'] = eq_df.prompt.apply(classify_prompt)

# 2. Вывод количества задач по каждому классу
print("=== Статистика по классам задач ===")
class_counts = eq_df['task_type'].value_counts()
print(class_counts.to_string())
print("\n")

# 3. Функция для решения и сравнения с эталонным ответом
def evaluate_solver(row):
    # Вызываем солвер
    result = solver.solve(row['prompt']) 
    
    # Строгая проверка типа: если это словарь (новая версия)
    if isinstance(result, dict):
        predicted = result.get('answer')
        debug_log = result.get('debug', [])
    # Если это строка или None (старая версия)
    else:
        predicted = result
        debug_log = ["Дебаг недоступен: используется старая версия солвера."]
    
    # Приводим оба ответа к строке и удаляем лишние пробелы для корректного сравнения
    true_answer = str(row['answer']).strip()
    
    # Обрабатываем None
    pred_answer = str(predicted).strip() if predicted is not None else "nan"
    
    is_correct = (true_answer == pred_answer)
    
    # Возвращаем 3 колонки
    return pd.Series(
        [pred_answer, is_correct, debug_log], 
        index=['predicted_answer', 'is_correct', 'debug_log']
    )

=== Статистика по классам задач ===
task_type
Cryptarithm (CSP)              823
AST Brute-force                660
Pseudo-Math (Format/String)     72




In [10]:
samples_per_class = 3

# Список уникальных классов в датафрейме
task_types = eq_df['task_type'].unique()

for task_type in task_types:
    print(f"\n{'='*5} Класс: {task_type} {'='*5}")
    
    # Фильтруем задачи текущего класса
    subset = eq_df[eq_df['task_type'] == task_type]
    
    # Берем первые N примеров (можно заменить .head() на .sample(), чтобы брать случайные)
    sample_df = subset.sample(samples_per_class)
    
    for idx, row in sample_df.iterrows():
        print(f"\nID: {row.get('id', idx)}")
        print(row['prompt'].strip())
        print(f"Ответ: {row['answer']}")
        print("-" * 5)


===== Класс: Cryptarithm (CSP) =====

ID: fa714b83
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
<^@%{ = $'
${@^{ = <<
>#)/# = >'/<
%'?($ = ^
#/?<< = '>
Now, determine the result for: ^{?#{
Ответ: /
-----

ID: d250fcc5
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
%{*!) = }''{
?)*)) = }'<)
%)*)% = {?!%
<{-)? = ?"
Now, determine the result for: '&-)<
Ответ: <%
-----

ID: a85864a9
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
[}+[} = [}[}
:)*}/ = :)}/
)}-%` = -/[
%\-:| = )\
(%*%/ = (%%/
Now, determine the result for: (\*}:
Ответ: (\}:
-----

===== Класс: AST Brute-force =====

ID: 9fa9ecdc
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
97%32 = 3103
25/84 = 109
31%85 = 2634
59/46 = 105
45/67 = 112
Now, determine the result for: 50^86

In [11]:
solver.solve("""In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
>:^@% = $]%@
@@}]# = :/
&]}]% = ]#
@$[$# = [%<
Now, determine the result for: $>}@/""")

{'answer': '>#<',
 'debug': ['--- Решение ---',
  'We model this as a cryptarithm: every visible non-operator symbol is one unique decimal digit, and multi-digit values cannot start with zero.',
  "Parsed examples: '>:^@%=$]%@', '@@}]#=:/', '&]}]%=]#', '@$[$#=[%<'",
  "Target expression: '$>}@/'",
  "Symbols to decode: '#', '$', '%', '&', '/', ':', '<', '>', '@', ']'",
  '',
  'Rule search',
  "Evaluating operator '['",
  'Examples: @$[$#=[%<',
  'Testing structurally possible combinations:',
  ' Config: standard',
  '  - sub_signed -> format (sign_pref_symbol_raw) [MATCH]',
  ' Config: little_endian',
  '  - sub_signed -> format (sign_pref_symbol_rev)',
  "Rule identified for '[': standard -> sub_signed -> sign_pref_symbol_raw",
  'Verifying examples for this operator:',
  '  @$ [ $# -> inputs A=26, B=67 -> 26 - 67 = -41 -> encode [%< [OK]',
  '',
  "Evaluating operator '^'",
  'Examples: >:^@%=$]%@',
  'Testing structurally possible combinations:',
  ' Config: standard',
  '  - mul -

In [12]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=24, progress_bar=True)


INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [13]:
import pandas as pd
df = pd.read_parquet('../../solver_results.parquet')
df

,id,prompt,answer,solver_answer,solver_correct,solver_type,solver_category,conditioned_on_answer,solver_mode,solver_ops,solver_mapping,solver_tier,solver_level,solver_radix,solver_numeric_answer,op_counts,min_op_count,num_ops
0,00457d26,"In Alice's Wonderland, a secret set of transfo...",@&,@&,True,arithmetic,arithmetic,True,standard,"{""*"": ""mul_p1"", ""-"": ""absdiff""}","{""!"": 6, ""\"""": 7, ""&"": 8, ""'"": 2, "">"": 0, ""@"":...",28.0,normal,10.0,18.0,"{""*"": 3, ""-"": 1}",1,2
1,00c032a8,"In Alice's Wonderland, a secret set of transfo...",\^?,\^?,True,arithmetic,little_endian,True,little_endian,"{""!"": ""mul"", ""]"": ""add"", ""<"": ""absdiff""}","{""#"": 6, ""&"": 8, ""("": 7, "")"": 1, ""?"": 2, ""@"": ...",12.0,normal,10.0,209.0,"{""]"": 1, ""<"": 1, ""!"": 2}",1,3
2,012cab1f,"In Alice's Wonderland, a secret set of transfo...",|@{,|@{,True,arithmetic,little_endian,True,little_endian,"{""'"": ""add"", "">"": ""absdiff"", ""]"": ""mul""}","{""\"""": 5, ""#"": 3, ""%"": 2, ""&"": 6, ""("": 9, "":"":...",12.0,normal,10.0,140.0,"{""]"": 1, "">"": 2, ""'"": 2}",1,3
3,0133bcec,"In Alice's Wonderland, a secret set of transfo...",\([#,\([#,True,concat,pure_concat,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""*"": 3, ""+"": 1, ""-"": 1}",1,3
4,017a871e,"In Alice's Wonderland, a secret set of transfo...",\:,\:,True,arithmetic,little_endian,True,little_endian,"{""-"": ""absdiff_m2"", ""+"": ""add_p2"", ""*"": ""a2_pl...","{""!"": 8, ""\"""": 7, ""#"": 2, "":"": 1, ""\\"": 3, ""]""...",47.0,deep,10.0,13.0,"{""+"": 1, ""-"": 1, ""*"": 1}",1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818,fdbdf50c,"In Alice's Wonderland, a secret set of transfo...","-\""","-\""",True,arithmetic,little_endian,True,little_endian,"{""-"": ""sub_signed"", ""}"": ""add"", ""("": ""mul""}","{""\"""": 2, ""%"": 0, ""'"": 7, "")"": 9, ""<"": 6, ""?"":...",12.0,normal,10.0,-23.0,"{""-"": 3, ""}"": 1, ""("": 1}",1,3
819,fe6da79d,"In Alice's Wonderland, a secret set of transfo...",:&]!,:&]!,True,arithmetic,mixed_concat,True,standard,"{""-"": ""neg_absdiff"", ""*"": ""mul"", ""+"": ""concat_...","{""!"": 0, ""#"": 4, ""$"": 3, ""&"": 5, ""'"": 2, "":"": ...",12.0,normal,10.0,1560.0,"{""+"": 1, ""-"": 3, ""*"": 1}",1,3
820,ff0e37ae,"In Alice's Wonderland, a secret set of transfo...","%'""`","%'""`",True,concat,pure_concat,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""*"": 2, ""+"": 1}",1,2
821,ff121f08,"In Alice's Wonderland, a secret set of transfo...",#<,#<,True,arithmetic,arithmetic,True,standard,"{""+"": ""sub_signed"", ""("": ""mul"", ""\\"": ""add_p1""}","{""!"": 3, ""#"": 1, ""%"": 4, ""/"": 5, ""<"": 2, "">"": ...",28.0,normal,10.0,12.0,"{""+"": 1, ""("": 2, ""\\"": 2}",1,3


In [14]:
print("Запуск солвера. Пожалуйста, подождите...")
results_df = eq_df.parallel_apply(evaluate_solver, axis=1)
eq_df = pd.concat([eq_df, results_df], axis=1)

# 5. Вывод процента решенных задач (accuracy) по каждому классу
print("\n=== Результаты решения по классам ===")
# Группируем по типу задачи и считаем среднее значение (True = 1, False = 0)
accuracy_stats = eq_df.groupby('task_type')['is_correct'].agg(['count', 'sum', 'mean'])

for task_type, row in accuracy_stats.iterrows():
    total = int(row['count'])
    correct = int(row['sum'])
    accuracy_percent = row['mean'] * 100
    
    print(f"Класс: {task_type}")
    print(f"Решено: {correct} из {total} ({accuracy_percent:.2f}%)\n")

# Итоговая точность по всему датасету
total_tasks = len(eq_df)
total_correct = eq_df['is_correct'].sum()
overall_accuracy = (total_correct / total_tasks) * 100 if total_tasks > 0 else 0

print(f"=== Общий итог ===")
print(f"Всего решено: {total_correct} из {total_tasks} ({overall_accuracy:.2f}%)")

Запуск солвера. Пожалуйста, подождите...



=== Результаты решения по классам ===
Класс: AST Brute-force
Решено: 511 из 660 (77.42%)

Класс: Cryptarithm (CSP)
Решено: 344 из 823 (41.80%)

Класс: Pseudo-Math (Format/String)
Решено: 72 из 72 (100.00%)

=== Общий итог ===
Всего решено: 927 из 1555 (59.61%)


In [15]:
failed_pseudo_math = eq_df[
    (eq_df['task_type'] == 'AST Brute-force') & 
    (eq_df['is_correct'] == False)
]

print(f"Всего нерешенных задач Cryptarithm (CSP): {len(failed_pseudo_math)}\n")

# Берем первые 10 примеров для анализа
sample_to_analyze = failed_pseudo_math.sample(30, random_state=4324)

for idx, row in sample_to_analyze.iterrows():
    print("="*60)
    print(f"ID: {row.get('id', idx)}")
    print(f"--- Prompt ---\n{row['prompt']}")
    #print(f"--- Answers ---")
    print(f"True Answer: {row['answer']}")
    #print(f"Predicted:   {row['predicted_answer']}")
    #print(f"DEBUG: {row['debug_log']}")


    maps = df[df["id"] == row.get("id", idx)][["solver_category", "solver_mapping", "op_counts"]].to_string()
    print(f"True answer simbol mapings (Мапинги для чисел)\n{maps}")
print("="*60)

Всего нерешенных задач Cryptarithm (CSP): 149

ID: c561264c
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
54-34 = 2
42-01 = 41
86-52 = 34
93-35 = -41
46*72 = 7246
Now, determine the result for: 79+02
True Answer: 0279
True answer simbol mapings (Мапинги для чисел)
Empty DataFrame
Columns: [solver_category, solver_mapping, op_counts]
Index: []
ID: 20f0fac9
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
78]18 = 6
95]24 = 23
52*94 = 145
21]91 = 7
Now, determine the result for: 31%96
True Answer: 2976
True answer simbol mapings (Мапинги для чисел)
Empty DataFrame
Columns: [solver_category, solver_mapping, op_counts]
Index: []
ID: 5c008804
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
98'21 = 101
07)76 = 0964
91'15 = 07
19@84 = 34
Now, determine the result

In [16]:
(83.08 +99.30+100.+100.+100.+59.61)/6

90.33166666666666

In [17]:
failed_pseudo_math = eq_df[
    (eq_df['task_type'] == 'Cryptarithm (CSP)') & 
    (eq_df['is_correct'] == True)
]

print(f"Всего решенных задач Cryptarithm (CSP): {len(failed_pseudo_math)}\n")

# Берем первые 10 примеров для анализа
sample_to_analyze = failed_pseudo_math.sample(30, random_state=312)

for idx, row in sample_to_analyze.iterrows():
    if ("Concat L+R" not in str(row['debug_log'])) and ("Concat R+L" not in str(row['debug_log'])):
        print("="*60)
        print(f"ID: {row.get('id', idx)}")
        print(f"--- Prompt ---\n{row['prompt']}")
        #print(f"--- Answers ---")
        print(f"True Answer: {row['answer']}")
        #print(f"Predicted:   {row['predicted_answer']}")
        #print(f"DEBUG: {row['debug_log']}")
print("="*60)

Всего решенных задач Cryptarithm (CSP): 344

ID: b4b73143
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
?{%[" = ?\!{
:!*"{ = ?##
/#*{# = #/
Now, determine the result for: >!%!/
True Answer: >\:
ID: 0b0a3643
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
#/-\@ = -@#
""+#) = )/
'#+/# = %"
\)-)@ = -'"
Now, determine the result for: '/-%)
True Answer: ""
ID: d0b1e41a
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
)#*)# = )#)#
)}-#" = -$
"$+`@ = "$`@
Now, determine the result for: #{*"!
True Answer: #{"!
ID: bca230fd
--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
$<:<^ = :]{
$'*\$ = \^'
')*]' = $)\)
<'*{$ = $^''
$$`]! = $$]!
Now, determine the result for: !<`[$
True Answer: !<[$


In [18]:
from src.solvers.equations.cryptarithm import CryptarithmCSPSolver

In [20]:
class_counts = eq_df['task_type']
class_counts

8       Cryptarithm (CSP)
26      Cryptarithm (CSP)
29        AST Brute-force
39      Cryptarithm (CSP)
41      Cryptarithm (CSP)
              ...        
9459      AST Brute-force
9466    Cryptarithm (CSP)
9467    Cryptarithm (CSP)
9482      AST Brute-force
9484    Cryptarithm (CSP)
Name: task_type, Length: 1555, dtype: str

In [21]:
analyze = eq_df[eq_df['task_type'] == "Cryptarithm (CSP)"]
analyze

,id,prompt,answer,prompt_eda,label,task_type,predicted_answer,is_correct,debug_log
8,00457d26,"In Alice's Wonderland, a secret set of transfo...",@&,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),@&,True,"[--- Решение ---, We model this as a cryptarit..."
26,00c032a8,"In Alice's Wonderland, a secret set of transfo...",\^?,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),nan,False,"[--- Решение ---, We model this as a cryptarit..."
39,012cab1f,"In Alice's Wonderland, a secret set of transfo...",|@{,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),nan,False,"[--- Решение ---, We model this as a cryptarit..."
41,0133bcec,"In Alice's Wonderland, a secret set of transfo...",\([#,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),\([#,True,"[--- Решение ---, We model this as a cryptarit..."
52,017a871e,"In Alice's Wonderland, a secret set of transfo...",\:,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),nan,False,"[--- Решение ---, We model this as a cryptarit..."
...,...,...,...,...,...,...,...,...,...
9415,fdbdf50c,"In Alice's Wonderland, a secret set of transfo...","-\""","In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),"\""",False,"[--- Решение ---, We model this as a cryptarit..."
9434,fe6da79d,"In Alice's Wonderland, a secret set of transfo...",:&]!,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),nan,False,[No solution survived all hypotheses and const...
9466,ff0e37ae,"In Alice's Wonderland, a secret set of transfo...","%'""`","In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),"%'""`",True,"[--- Решение ---, We model this as a cryptarit..."
9467,ff121f08,"In Alice's Wonderland, a secret set of transfo...",#<,"In Alice's Wonderland, a secret set of transfo...",equations transformation,Cryptarithm (CSP),#<,True,"[--- Решение ---, We model this as a cryptarit..."


In [22]:
def analyze_prompt(prompt: str) -> str:
    examples, target = solver._extract_sections(prompt)
    results = []
    for line in examples.strip().split('\n'):
        if '=' in line:
            lhs_raw, rhs_raw = line.split('=', 1)
            results.append(lhs_raw[2])
    return results


def analyze_prompt_all(prompt: str) -> str:
    examples, target = solver._extract_sections(prompt)
    results = []
    for line in examples.strip().split('\n'):
        if '=' in line:
            lhs_raw, rhs_raw = line.split('=', 1)
            results.append([i for i in lhs_raw] + [i for i in rhs_raw])
    return results

In [23]:
results = analyze.prompt.apply(analyze_prompt_all)

In [24]:
set([v for j in results.to_list() for x in j for v in x])

{' ',
 '!',
 '"',
 '#',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 '-',
 '/',
 ':',
 '<',
 '>',
 '?',
 '@',
 '[',
 '\\',
 ']',
 '^',
 '`',
 '{',
 '|',
 '}'}

In [25]:
len(set([v for j in results.to_list() for x in j for v in x]))

27

In [26]:

for v in set([v for j in results.to_list() for x in j for v in x]):
    print(f"{v} = {ord(v)}")

" = 34
# = 35
| = 124
' = 39
< = 60
/ = 47
\ = 92
* = 42
  = 32
[ = 91
: = 58
+ = 43
- = 45
! = 33
% = 37
@ = 64
> = 62
` = 96
$ = 36
) = 41
^ = 94
] = 93
} = 125
{ = 123
& = 38
( = 40
? = 63


In [27]:
result = list(set([v for j in results.to_list() for x in j for v in x]))

In [28]:
result.sort()

In [29]:
result

[' ',
 '!',
 '"',
 '#',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 '-',
 '/',
 ':',
 '<',
 '>',
 '?',
 '@',
 '[',
 '\\',
 ']',
 '^',
 '`',
 '{',
 '|',
 '}']

In [30]:
d = {}
for i, r in enumerate(result[1:]):
    d[r] = i+1
    

In [31]:
print(d)

{'!': 1, '"': 2, '#': 3, '$': 4, '%': 5, '&': 6, "'": 7, '(': 8, ')': 9, '*': 10, '+': 11, '-': 12, '/': 13, ':': 14, '<': 15, '>': 16, '?': 17, '@': 18, '[': 19, '\\': 20, ']': 21, '^': 22, '`': 23, '{': 24, '|': 25, '}': 26}
